# PIC_SUB_04: manuscript outputs of the PIC external validation

Rebuilds, from the exact output tables of PIC_SUB_02 and PIC_SUB_03, everything the manuscript reports about the external
validation in PIC: every number in the text, Figure 7 with its legend, and Additional file 2, Tables S27-S33. Nothing is typed in
by hand. The notebook reads aggregate tables only; it does not read PIC itself.

**Where to put it.** In the PIC_SUB_02 folder, next to:

* `output_pic_sub02_LOCAL_ONLY/` and `output_pic_sub02/` (PIC_SUB_02, the analysis of record)
* `output_pic_sub03_posthoc_LOCAL_ONLY/` (PIC_SUB_03, the post hoc analysis P-1; `output_pic_sub03_posthoc/` is used if present)
* `PIC_SUB_02_frozen_wuhan_model.json`
* optional: `Additional_file_2_supplementary_tables.xlsx` of the submitted manuscript (Tables S1-S26); Tables S27-S33 are then
  appended to a copy of it. Without it, a workbook with Tables S27-S33 only is written.
* optional: `Manuscript.docx`; every PIC number of the text is then looked up in it.

**How to run.** *Restart Kernel and Run All Cells* (under a minute). Needs pandas, NumPy, Matplotlib and openpyxl (python-docx
for the optional manuscript check). Figure 7 uses Arial when it is installed.

**Outputs** in `output_pic_sub04_manuscript/`: `pic_numbers.json`, `manuscript_numbers.csv`, `Figure7_external_validation.pdf`
and `.png`, `figure7_legend.md`, `Additional_file_2_supplementary_tables.xlsx` and `run_info_sub04.json`. The last cell compares
the results with those of the manuscript build (MATCH or DIFFERENT for each).

**Sharing.** The outputs hold only aggregate results that the manuscript reports. The `*_LOCAL_ONLY` input folders stay on this
computer; only this notebook (outputs cleared, or as run) goes to the code repository.

In [ ]:
# Setup: input folders, checks of the inputs, output folder.
NOTEBOOK_MARKER = 'PIC_SUB_04_MANUSCRIPT_OUTPUTS_NOTEBOOK'
import hashlib, json, os, platform, re, shutil, sys
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import openpyxl

HERE = Path.cwd()
LOC = HERE / 'output_pic_sub02_LOCAL_ONLY'                 # exact tables of PIC_SUB_02 (analysis of record)
SHARE = HERE / 'output_pic_sub02'                           # shareable tables of PIC_SUB_02 (2-decimal correlations)
SUB03_LOCAL = HERE / 'output_pic_sub03_posthoc_LOCAL_ONLY'  # exact tables of PIC_SUB_03 (post hoc P-1)
MODEL_FILE = HERE / 'PIC_SUB_02_frozen_wuhan_model.json'
AF2_IN = HERE / 'Additional_file_2_supplementary_tables.xlsx'
DOCX = HERE / 'Manuscript.docx'
OUT = HERE / 'output_pic_sub04_manuscript'

for p in (LOC, SHARE, SUB03_LOCAL, MODEL_FILE):
    if not p.exists():
        raise FileNotFoundError(f'{p.name} not found in {HERE}. Put this notebook in the PIC_SUB_02 folder (see the first cell).')
FP_SUB02 = '6193ed4ce89128519eb66ab93dd5fd1c16044a152e205a8c0217c027d1e42907'
FP_SUB03 = 'de0bea0faebb57565bb956b203f43119a74f47204f509be9ae1b0b37fd98fdc9'
MODEL_SHA = 'a8a51287d7dd2cb67b623328ffedc663deb9a01b3c5364db3c4e9d8f39be2179'
RUN = json.loads((LOC / 'run_info.json').read_text(encoding='utf-8'))
RUN3 = json.loads((SUB03_LOCAL / 'run_info.json').read_text(encoding='utf-8'))
assert RUN['code_sha256'] == FP_SUB02 and RUN['gate_met'] and not RUN['synthetic_fixture'], 'output_pic_sub02_LOCAL_ONLY is not the analysis of record'
assert RUN3['code_sha256'] == FP_SUB03 and not RUN3['synthetic_fixture'], 'output_pic_sub03_posthoc_LOCAL_ONLY is not from PIC_SUB_03'
assert hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest() == MODEL_SHA, 'the frozen model file differs from the deposited one'
same = [f for f in sorted(p.name for p in LOC.glob('[01]*.csv')) if (SUB03_LOCAL / f).read_bytes() != (LOC / f).read_bytes()]
assert not same, f'PIC_SUB_03 tables differ from PIC_SUB_02: {same}'
PKG = json.loads(MODEL_FILE.read_text(encoding='utf-8'))
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()
print('Inputs checked: PIC_SUB_02 6193ed4c..., PIC_SUB_03 de0bea0f... (tables 01-14 identical), frozen model a8a51287...')
print('python', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__, '| matplotlib', matplotlib.__version__,
      '| openpyxl', openpyxl.__version__)

## Every PIC number of the manuscript
Read from the exact tables of PIC_SUB_02 and PIC_SUB_03 and from the frozen model file, with consistency checks; written to
`pic_numbers.json`.

In [ ]:
def t(name):
    return pd.read_csv(LOC / name, dtype=str, keep_default_na=False)


def ci3(s):
    """'8.74 (3.92-20.90)' -> (8.74, 3.92, 20.90)."""
    m = re.fullmatch(r'\s*(-?[\d.]+) \((-?[\d.]+)-(-?[\d.]+)\)\s*', s)
    return tuple(float(x) for x in m.groups()) if m else None


def med_iqr(s):
    m = re.fullmatch(r'([\d.]+) \(([\d.]+)-([\d.]+)\)', s)
    return tuple(float(x) for x in m.groups())


N = {}
# flow
fl = t('02_flow.csv')
N['flow'] = {int(r.step): int(r.count_n) for r in fl.itertuples()}
# cohort
cb = t('03b_cohort.csv').set_index('measure').value
N['cohort'] = {'n': int(cb.children_n), 'age_median_iqr': med_iqr(cb.age_years_median_iqr), 'male_n': int(cb.male_n),
               'male_pct': float(cb.male_pct), 'deaths_n': int(cb.in_hospital_deaths_n), 'deaths_pct': float(cb.in_hospital_deaths_pct),
               'picu_n': int(cb.first_icu_unit_picu_n), 'picu_pct': float(cb.first_icu_unit_picu_pct),
               'tests_median_iqr': med_iqr(cb.blood_tests_of_22_median_iqr), 'missing_pct': float(cb.missing_pct_24_indicators)}
cg = t('03_code_groups.csv')
N['code_groups'] = {r.code_group: (int(r.children_n), float(r.children_pct)) for r in cg.itertuples()}
# classes
cl = t('06_classes.csv')
N['classes'] = {r['class']: {'n': int(r.children_n), 'pct': float(r.children_pct), 'share_pct': float(r.estimated_share_pct),
                             'mean_post': float(r.mean_posterior_assigned), 'post_below_08_n': int(r.posterior_below_0_8_n),
                             'r': float(r.profile_r_vs_wuhan_same_class), 'wuhan_pct': float(r.wuhan_children_pct),
                             'wuhan_picu_pct': float(r.wuhan_picu_children_pct)} for _, r in cl.iterrows()}
# primary (sensitivity table, primary row, 3 decimals)
se = t('13_sensitivity.csv')
N['sensitivity'] = {}
for _, r in se.iterrows():
    key = r.analysis.split(':')[0]
    N['sensitivity'][key] = {'label': r.analysis, 'n': int(r.children_n), 's1_n': int(r.S1_n), 's1_pct': float(r.S1_pct),
                             'r_ci': ci3(r.S1_profile_r_ci), 'met': r.replication_criterion_met == 'yes', 'il6': ci3(r.il6_ratio_S1_vs_S6),
                             'il10': ci3(r.il10_ratio_S1_vs_S6), 'rr': ci3(r.death_rr_S1_vs_others),
                             'kappa': float(r.kappa_vs_primary) if r.kappa_vs_primary else None,
                             'common_n': int(r.common_children_n) if r.common_children_n else None}
P = N['sensitivity']['Primary']
N['primary'] = {'s1_n': P['s1_n'], 's1_pct': P['s1_pct'], 'r_ci': P['r_ci'], 'met': P['met'], 'n': P['n']}
# validators
vc = t('09_validator_contrasts.csv')
N['validators'] = {f'{r.validator}|{r.contrast}': {'s1_measured': int(r.S1_measured_n), 'comp_measured': int(r.comparison_measured_n),
                                                    'ratio_ci': ci3(r.ratio_of_geometric_means_ci),
                                                    'wuhan_modal': float(r.wuhan_ratio_modal)} for _, r in vc.iterrows()}
vb = t('08_validators_by_class.csv')
N['validator_measured'] = {r.validator: (int(r.measured_n), float(r.measured_pct)) for r in vb.itertuples()}
N['validator_gm'] = {r.validator: {c: (int(getattr(r, f'{c}_measured_n')), float(getattr(r, f'{c}_geometric_mean')) if getattr(r, f'{c}_geometric_mean') else None)
                                   for c in ['S1', 'S2', 'S3', 'S4', 'S5', 'S6']} for r in vb.itertuples()}
# death
de = t('10_death.csv')
N['death'] = {}
for _, r in de.iterrows():
    if r.row in ('S1', 'S2', 'S3', 'S4', 'S5', 'S6'):
        N['death'][r.row] = {'n': int(r.children_n), 'deaths': int(r.deaths_n), 'pct_ci': ci3(r.estimate_ci),
                             'wuhan_ihd_pct': float(r.wuhan_in_hospital_death_pct)}
    elif r.row.startswith('Risk ratio, S1 vs all'):
        N['death']['rr_s1_vs_others'] = ci3(r.estimate_ci)
    elif r.row.startswith('Odds ratio'):
        N['death']['or_age_adjusted'] = ci3(r.estimate_ci)
    elif r.row.startswith('Risk ratio, S1 vs S6'):
        N['death']['rr_s1_vs_s6'] = r.estimate_ci
others_n = N['cohort']['n'] - N['death']['S1']['n']
others_d = N['cohort']['deaths_n'] - N['death']['S1']['deaths']
assert others_n == sum(N['death'][c]['n'] for c in ['S2', 'S3', 'S4', 'S5', 'S6'])
assert others_d == sum(N['death'][c]['deaths'] for c in ['S2', 'S3', 'S4', 'S5', 'S6'])
N['death']['others'] = {'n': others_n, 'deaths': others_d, 'pct': round(100 * others_d / others_n, 1)}
# bedside rule
br = t('11_bedside_rule.csv').set_index('measure')
B = {'assessable': int(br.loc['Children assessable (AST and LDH measured)', 'value_n']),
     'tp': int(br.loc['Rule positive and S1 (true positive)', 'value_n']), 'fn': int(br.loc['Rule negative and S1 (false negative)', 'value_n']),
     'fp': int(br.loc['Rule positive, not S1 (false positive)', 'value_n']), 'tn': int(br.loc['Rule negative, not S1 (true negative)', 'value_n']),
     'sens': ci3(br.loc['Sensitivity (%)', 'estimate_ci']), 'spec': ci3(br.loc['Specificity (%)', 'estimate_ci']),
     'ppv': ci3(br.loc['Positive predictive value (%)', 'estimate_ci']), 'npv': ci3(br.loc['Negative predictive value (%)', 'estimate_ci']),
     'pos_n': int(br.loc['Rule positive: children', 'value_n']), 'pos_d': int(br.loc['Rule positive: in-hospital deaths', 'value_n']),
     'neg_n': int(br.loc['Rule negative: children', 'value_n']), 'neg_d': int(br.loc['Rule negative: in-hospital deaths', 'value_n']),
     'rr': ci3(br.loc['In-hospital death, risk ratio rule positive vs negative', 'estimate_ci']),
     'pos_pct': float(br.loc['In-hospital death, rule positive (%)', 'estimate_ci']),
     'neg_pct': float(br.loc['In-hospital death, rule negative (%)', 'estimate_ci'])}
assert B['tp'] + B['fn'] + B['fp'] + B['tn'] == B['assessable'] == B['pos_n'] + B['neg_n']
assert B['tp'] + B['fp'] == B['pos_n']
B['s1_assessable'] = B['tp'] + B['fn']
B['s1_not_assessable'] = N['primary']['s1_n'] - B['s1_assessable']
N['bedside_pic'] = B
W = PKG['bedside_rule']['wuhan_performance']
N['bedside_wuhan'] = {'all': W['all'], '2019-2022': W['2019-2022'], '2023-2026': W['2023-2026'],
                      'docd': W['death_or_critical_discharge'], 'ihd': W['in_hospital_death']}
# model fit
mf = t('12_model_fit.csv').set_index('measure')
N['fit'] = {'ll_mean': float(mf.loc['Log-likelihood per observed value, mean', 'pic']),
            'll_mean_wuhan': float(mf.loc['Log-likelihood per observed value, mean', 'wuhan']),
            'below_p5_n': int(mf.loc['Children below the Wuhan 5th percentile of log-likelihood per value (children, rounded)', 'pic']),
            'post_mean': float(mf.loc['Largest posterior probability, mean', 'pic']),
            'post_mean_wuhan': float(mf.loc['Largest posterior probability, mean', 'wuhan']),
            'post_below_08_n': int(mf.loc['Largest posterior probability below 0.8 (children, rounded)', 'pic']),
            'missing_pct': float(mf.loc['Missing indicator values (%)', 'pic']),
            'missing_pct_wuhan': float(mf.loc['Missing indicator values (%)', 'wuhan'])}
# indicator availability (exact measured counts)
sc = t('12b_indicator_scale.csv')
N['indicator_measured'] = {r.indicator: int(r.measured_n) for r in sc.itertuples()}
# profiles (PIC observed means and Wuhan model means of every class)
pr = t('07_profiles.csv')
N['profiles'] = {r.indicator: {k: (float(v) if v not in ('', 'suppressed') else None) for k, v in r._asdict().items()
                               if k.startswith(('pic_S', 'wuhan_S'))} for r in pr.itertuples()}
# de novo LPA
dn = t('14_de_novo_lpa.csv')
fits = dn[dn.fitted.eq('True')]
N['de_novo'] = {'chosen_k': int(fits[fits.chosen.eq('True')].classes.iloc[0]),
                'bic': {int(r.classes): float(r.bic) for r in fits.itertuples()},
                'classes': [{'n': int(r.children_n), 'r_s1': float(r.r_with_wuhan_S1), 'closest': r.closest_wuhan_class,
                             'r_closest': float(r.r_with_closest_wuhan_class), 'also_s1': int(r.also_assigned_S1_by_frozen_model_n)}
                            for r in dn[dn.classes.str.startswith('chosen solution')].itertuples()]}
# Wuhan reference values from the model file
WR = PKG['wuhan_reference']
N['wuhan'] = {'S1_pct': WR['S1_pct'], 'S1_pct_picu': WR['S1_pct_picu'], 'r_S1_vs_S2': WR['profile_r_S1_vs_other_wuhan_classes']['S2'],
              'validators': {k: {'ratio_S1_vs_S6': v['ratio_S1_vs_S6'], 'ratio_S1_vs_others': v['ratio_S1_vs_others'],
                                 'gm': v['geometric_mean_by_class']} for k, v in WR['validators'].items()},
              'ihd': WR['in_hospital_death'], 'check': PKG['checks_in_wuhan']['external_preprocessing_applied_to_wuhan']}
# post hoc P-1: exact tables 15 and 15b of PIC_SUB_03
P1L = SUB03_LOCAL
q = pd.read_csv(P1L / '15_posthoc_discharge_only.csv', dtype=str, keep_default_na=False)
qb = pd.read_csv(P1L / '15b_posthoc_counts.csv', dtype=str, keep_default_na=False).set_index('measure').value
assert q.iloc[0].children_n == str(N['cohort']['n']) and q.iloc[0].S1_profile_r_ci == se.iloc[0].S1_profile_r_ci   # same primary
r1 = q.iloc[1]
N['p1'] = {'n': int(r1.children_n), 's1_n': int(r1.S1_n), 's1_pct': float(r1.S1_pct), 'r_ci': ci3(r1.S1_profile_r_ci),
           'met': r1.replication_criterion_met == 'yes', 'il6': ci3(r1.il6_ratio_S1_vs_S6), 'il10': ci3(r1.il10_ratio_S1_vs_S6),
           'rr': ci3(r1.death_rr_S1_vs_others), 'counts': {k: int(v) for k, v in qb.items()}}
N['_source'] = {'run_info': RUN, 'local_folder': 'output_pic_sub02_LOCAL_ONLY'}
(OUT / 'pic_numbers.json').write_text(json.dumps(N, indent=1, ensure_ascii=False), encoding='utf-8', newline='\n')
print('pic_numbers.json written:', ', '.join(k for k in N if not k.startswith('_')))

## The numbers as printed in the manuscript
Each sentence fragment of the manuscript that carries a PIC number, rebuilt from the tables with the manuscript's rounding rules
(ratios to 3 significant digits and at most 2 decimals; correlations to 2 decimals as the analysis notebook rounded them). Written
to `manuscript_numbers.csv`; if `Manuscript.docx` is in the folder, each fragment is looked up in its text.

In [ ]:
N = json.loads((OUT / 'pic_numbers.json').read_text(encoding='utf-8'))      # as saved (JSON lists), like the manuscript build
# Every PIC sentence fragment of the manuscript that carries a number, rebuilt from the tables: (section, text as printed).
# Needs N (pic_numbers.json, loaded), PKG (frozen model file) and SHARE (the shareable PIC_SUB_02 output folder).
from decimal import Decimal, ROUND_HALF_UP


def sig3(x):
    """3 significant digits, at most 2 decimals, half up on the digits reported in the tables (the manuscript's rule for ratios)."""
    d = Decimal(str(x))
    return str(d.quantize(Decimal(1).scaleb(max(d.adjusted() - 2, -2)), rounding=ROUND_HALF_UP))


def d1(x):
    return str(Decimal(str(x)).quantize(Decimal('0.1'), rounding=ROUND_HALF_UP))


def d2(x):
    return str(Decimal(str(x)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP))


def pct(k, n):
    return d1(Decimal(100 * k) / Decimal(n))


def share_r(analysis_prefix):
    """Profile r with its CI to 2 decimals, as the notebook rounded it from full precision (shareable table)."""
    s = pd.read_csv(SHARE / '13_sensitivity.csv', dtype=str, keep_default_na=False)
    r_ = s[s.analysis.str.startswith(analysis_prefix)].S1_profile_r_ci
    assert len(r_) == 1, analysis_prefix
    m = re.fullmatch(r'([\d.]+) \(([\d.]+)-([\d.]+)\)', r_.iloc[0])
    return m.groups()


NUMWORD = {2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six'}
fl, co, cg, cl = N['flow'], N['cohort'], N['code_groups'], N['classes']
de, bp, ft, sens, val = N['death'], N['bedside_pic'], N['fit'], N['sensitivity'], N['validators']
bw = PKG['bedside_rule']['wuhan_performance']
wr = PKG['wuhan_reference']
assert fl['4'] == fl['5'], 'children aged 18 or older were excluded; the text does not mention them'
r_prim = share_r('Primary')
E = []

# Abstract
E.append(('Abstract', f"In PIC, {cl['S1']['n']} of {co['n']} children ({d1(cl['S1']['pct'])}%) were assigned to it; their "
          f"profile correlated {r_prim[0]} ({r_prim[1]}–{r_prim[2]}) with the derivation profile, meeting the replication criterion."))
i10 = val['IL-10|S1 vs S6 (least abnormal)']['ratio_ci']
i6 = val['IL-6|S1 vs S6 (least abnormal)']['ratio_ci']
E.append(('Abstract', f"Their IL-10 and IL-6 were {sig3(i10[0])}-fold ({sig3(i10[1])}–{sig3(i10[2])}) and {sig3(i6[0])}-fold "
          f"({sig3(i6[1])}–{sig3(i6[2])}) higher than in the least abnormal subphenotype, "
          f"in-hospital mortality was {d1(de['S1']['pct_ci'][0])}% versus {d1(de['others']['pct'])}% in other children (risk ratio "
          f"{sig3(de['rr_s1_vs_others'][0])}, {sig3(de['rr_s1_vs_others'][1])}–{sig3(de['rr_s1_vs_others'][2])}), and the bedside rule had "
          f"sensitivity {d1(bp['sens'][0])}% and specificity {d1(bp['spec'][0])}%."))
E.append(('Abstract', f"identified it with sensitivity {d1(100 * bw['all']['sensitivity'][0])}% and specificity "
          f"{d1(100 * bw['all']['specificity'][0])}%."))

# Methods
chk = PKG['checks_in_wuhan']['external_preprocessing_applied_to_wuhan']
E.append(('Methods', f"reproduced the original assignment in {chk['agreement_n']} of {chk['children']} children "
          f"({pct(chk['agreement_n'], chk['children'])}%; κ {d2(chk['kappa'])}), including all {chk['S1_kept']} hyperinflammatory children"))
E.append(('Methods', f"The threshold is well above the correlation of {d2(wr['profile_r_S1_vs_other_wuhan_classes']['S2'])} between the "
          f"derivation hyperinflammatory profile and the closest other class."))
assert max(wr['profile_r_S1_vs_other_wuhan_classes'].items(), key=lambda kv: kv[1])[0] == 'S2'

# Results: bedside rule in the derivation cohort
a = bw['all']
E.append(('Results', f"Of the {a['assessable']} children with AST and LDH measured, the rule was positive in {a['tp'] + a['fp']}, including "
          f"{a['tp']} of the {a['tp'] + a['fn']} hyperinflammatory children; its apparent sensitivity was {d1(100 * a['sensitivity'][0])}% "
          f"(95% CI {d1(100 * a['sensitivity'][1])}–{d1(100 * a['sensitivity'][2])}), specificity {d1(100 * a['specificity'][0])}% "
          f"({d1(100 * a['specificity'][1])}–{d1(100 * a['specificity'][2])}), positive predictive value {d1(100 * a['ppv'][0])}% "
          f"({d1(100 * a['ppv'][1])}–{d1(100 * a['ppv'][2])}) and negative predictive value {d1(100 * a['npv'][0])}% "
          f"({d1(100 * a['npv'][1])}–{d1(100 * a['npv'][2])}; Figure 7F)."))
late, dd = bw['2023-2026'], bw['death_or_critical_discharge']
assert late['fn'] == 0
E.append(('Results', f"In 2023–2026 it identified all {late['tp']} hyperinflammatory children, with specificity {d1(100 * late['specificity'][0])}%."))
E.append(('Results', f"Death or critical discharge occurred in {dd['rule_positive_events']} of {dd['rule_positive']} rule-positive children "
          f"({pct(dd['rule_positive_events'], dd['rule_positive'])}%) and {dd['rule_negative_events']} of {dd['rule_negative']} rule-negative "
          f"children ({pct(dd['rule_negative_events'], dd['rule_negative'])}%; risk ratio {sig3(dd['risk_ratio'][0])}, 95% CI "
          f"{sig3(dd['risk_ratio'][1])}–{sig3(dd['risk_ratio'][2])})."))

# Results: PIC cohort
E.append(('Results', f"Of {fl['1']} PIC admissions with an inclusion code, {fl['2']} were a child's first; {fl['2'] - fl['3']} children had "
          f"an exclusion code and {fl['3'] - fl['4']} were younger than 28 days, and {fl['6']} of the remaining {fl['5']} had at least 11 "
          f"of the 22 tests"))
age = co['age_median_iqr']
E.append(('Results', f"Their median age was {age[0]:.2f} years (IQR {age[1]:.2f}–{age[2]:.2f}), {co['male_n']} ({d1(co['male_pct'])}%) were "
          f"boys, {co['picu_n']} ({d1(co['picu_pct'])}%) had the PICU as their first intensive care unit, and {co['deaths_n']} "
          f"({d1(co['deaths_pct'])}%) died in hospital."))
G = {'unspecified meningoencephalitis or encephalitis': 'Meningoencephalitis or encephalitis, unspecified', 'viral encephalitis': 'Viral encephalitis',
     'acute encephalopathy': 'Encephalopathy codes', 'acute disseminated encephalomyelitis': 'ADEM or disseminated encephalitis',
     'brainstem or focal encephalitis': 'Brainstem or focal encephalitis, encephalomyelitis', 'bacterial meningoencephalitis': 'Bacterial meningoencephalitis'}
order = sorted(G, key=lambda g: -cg[G[g]][0])
assert order[0] == 'unspecified meningoencephalitis or encephalitis'
parts = [f"{g} in {cg[G[g]][0]} children ({d1(cg[G[g]][1])}%)" if i == 0 else f"{g} in {cg[G[g]][0]} ({d1(cg[G[g]][1])}%)"
         for i, g in enumerate(['unspecified meningoencephalitis or encephalitis', 'viral encephalitis', 'acute encephalopathy',
                                'acute disseminated encephalomyelitis', 'brainstem or focal encephalitis', 'bacterial meningoencephalitis'])]
E.append(('Results', 'Each child was counted in one diagnosis code group, the first that matched in a fixed order (Additional file 2: '
          'Table S27): ' + ', '.join(parts[:-1]) + ' and ' + parts[-1] + '.'))
tm = co['tests_median_iqr']
E.append(('Results', f"The median number of tests measured was {tm[0]:g} of 22 (IQR {tm[1]:g}–{tm[2]:g})."))
im = N['indicator_measured']
miss = {k: co['n'] - v for k, v in im.items()}
tot = sum(miss.values())
assert d1(Decimal(100 * tot) / Decimal(co['n'] * len(im))) == d1(co['missing_pct'])
grp = ['Temperature', 'Heart rate (z-score for age)', 'Respiratory rate (z-score for age)', 'Procalcitonin', 'Fibrinogen', 'D-dimer']
share_vit = pct(sum(miss[k] for k in grp), tot)
E.append(('Results', f"Of the indicator values, {d1(co['missing_pct'])}% were missing ({d1(ft['missing_pct_wuhan'])}% in the derivation "
          f"cohort); {share_vit}% of the missing values were vital signs, charted in PIC only in intensive care, or procalcitonin, "
          f"fibrinogen and D-dimer"))

# Results: subphenotypes
E.append(('Results', f"The frozen model assigned {cl['S1']['n']} children ({d1(cl['S1']['pct'])}%) to the hyperinflammatory subphenotype, "
          f"compared with {d1(wr['S1_pct'])}% of all derivation children and {d1(wr['S1_pct_picu'])}% of those admitted to the PICU"))
E.append(('Results', f"Their mean profile correlated {r_prim[0]} (95% CI {r_prim[1]}–{r_prim[2]}) with the derivation profile, meeting the "
          f"replication criterion"))
mid = [cl[c]['r'] for c in ['S2', 'S3', 'S4', 'S5']]
assert d2(cl['S6']['r']) == '0.80' and cl['S6']['r'] >= 0.80 and max(mid) < 0.80
E.append(('Results', f"The least abnormal subphenotype also matched its derivation profile (r = {d2(cl['S6']['r'])}), and the four middle "
          f"subphenotypes matched less closely (r {min(mid):.3f} to {max(mid):.3f}; Figure 7B)."))

# Results: cytokines and death
vm = N['validator_measured']
E.append(('Results', f"IL-6 and IL-10 were measured in {vm['IL-6'][0]} ({d1(vm['IL-6'][1])}%) and {vm['IL-10'][0]} ({d1(vm['IL-10'][1])}%) children."))
E.append(('Results', f"IL-10 was {sig3(i10[0])}-fold higher in the hyperinflammatory subphenotype (95% CI {sig3(i10[1])}–{sig3(i10[2])}; "
          f"derivation cohort, by most likely class, {sig3(val['IL-10|S1 vs S6 (least abnormal)']['wuhan_modal'])}) and IL-6 {sig3(i6[0])}-fold "
          f"higher ({sig3(i6[1])}–{sig3(i6[2])}; derivation {sig3(val['IL-6|S1 vs S6 (least abnormal)']['wuhan_modal'])})"))
o10 = val['IL-10|S1 vs all other subphenotypes']['ratio_ci']
o6 = val['IL-6|S1 vs all other subphenotypes']['ratio_ci']
E.append(('Results', f"Against all other subphenotypes, the ratios were {sig3(o10[0])} ({sig3(o10[1])}–{sig3(o10[2])}) for IL-10 and "
          f"{sig3(o6[0])} ({sig3(o6[1])}–{sig3(o6[2])}) for IL-6."))
tnf = val['TNF-alpha|S1 vs S6 (least abnormal)']['ratio_ci']
E.append(('Results', f"TNF-α differed little (ratio {sig3(tnf[0])}, {sig3(tnf[1])}–{sig3(tnf[2])}), and ferritin was measured in only "
          f"{vm['Ferritin'][0]} children"))
s1d, rr, orr = de['S1'], de['rr_s1_vs_others'], de['or_age_adjusted']
assert de['S5']['deaths'] == 0 and de['S6']['deaths'] == 0
E.append(('Results', f"In-hospital death occurred in {s1d['deaths']} of {s1d['n']} hyperinflammatory children ({d1(s1d['pct_ci'][0])}%, 95% CI "
          f"{d1(s1d['pct_ci'][1])}–{d1(s1d['pct_ci'][2])}) and {de['others']['deaths']} of {de['others']['n']} others "
          f"({d1(de['others']['pct'])}%; risk ratio {sig3(rr[0])}, 95% CI {sig3(rr[1])}–{sig3(rr[2])}; odds ratio adjusted for age "
          f"{sig3(orr[0])}, {sig3(orr[1])}–{sig3(orr[2])}), and in none of the {de['S5']['n'] + de['S6']['n']} children in the low metabolic "
          f"stress and least abnormal subphenotypes"))

# Results: bedside rule in PIC
E.append(('Results', f"Of {bp['assessable']} children with AST and LDH measured, the rule was positive in {bp['pos_n']}, including {bp['tp']} of "
          f"the {bp['tp'] + bp['fn']} assessable hyperinflammatory children: sensitivity {d1(bp['sens'][0])}% (95% CI {d1(bp['sens'][1])}–"
          f"{d1(bp['sens'][2])}), specificity {d1(bp['spec'][0])}% ({d1(bp['spec'][1])}–{d1(bp['spec'][2])}), positive predictive value "
          f"{d1(bp['ppv'][0])}% ({d1(bp['ppv'][1])}–{d1(bp['ppv'][2])}) and negative predictive value {d1(bp['npv'][0])}% "
          f"({d1(bp['npv'][1])}–{d1(bp['npv'][2])}) (Figure 7F)."))
E.append(('Results', f"In-hospital death occurred in {bp['pos_d']} of {bp['pos_n']} rule-positive children ({d1(bp['pos_pct'])}%) and "
          f"{bp['neg_d']} of {bp['neg_n']} rule-negative children ({d1(bp['neg_pct'])}%; risk ratio {sig3(bp['rr'][0])}, 95% CI "
          f"{sig3(bp['rr'][1])}–{sig3(bp['rr'][2])})."))

# Results: sensitivity analyses, de novo model, P-1
tested = ['S-a', 'S-b', 'S-c', 'S-d', 'S-g']
assert all(sens[k]['met'] for k in tested)
r2s = [Decimal(share_r(k + ':')[0]) for k in tested]
kap = [sens[k]['kappa'] for k in tested if sens[k]['kappa'] is not None]
E.append(('Results', f"The replication criterion was met in all {NUMWORD[len(tested)]} sensitivity analyses that tested it, with "
          f"{min(sens[k]['s1_n'] for k in tested)} to {max(sens[k]['s1_n'] for k in tested)} children assigned to the hyperinflammatory "
          f"subphenotype, profile correlations of {min(r2s)} to {max(r2s)} and κ with the primary assignment of {d2(min(kap))} to {d2(max(kap))}"))
se_ = sens['S-e']
E.append(('Results', f"Of the {se_['n']} children with encephalopathy codes, {se_['s1_n']} were hyperinflammatory (r = {share_r('S-e:')[0]})."))
dn = N['de_novo']
s1c = [c for c in dn['classes'] if c['closest'] == 'S1' and c['r_s1'] >= 0.80]
assert len(s1c) == 1 and s1c[0]['also_s1'] == cl['S1']['n']
E.append(('Results', f"A de novo latent profile model fitted in PIC alone chose {NUMWORD[dn['chosen_k']]} classes; one, of {s1c[0]['n']} children, "
          f"correlated {d2(s1c[0]['r_s1'])} with the derivation hyperinflammatory profile and contained all {cl['S1']['n']} children"))
p1 = N['p1']
left = p1['counts']['children_with_inclusion_code_only_among_admitting_or_clinic_diagnoses_n']
assert left == 1 and p1['counts']['S1_children_left_out_n'] == 0 and p1['counts']['in_hospital_deaths_left_out_n'] == 0
p1r = pd.read_csv(SHARE.parent / 'output_pic_sub03_posthoc' / '15_posthoc_discharge_only.csv', dtype=str, keep_default_na=False) \
    if (SHARE.parent / 'output_pic_sub03_posthoc').is_dir() else None
p1_r2 = (re.match(r'([\d.]+)', p1r.iloc[1].S1_profile_r_ci).group(1) if p1r is not None else d2(p1['r_ci'][0]))
E.append(('Results', f"one child, not hyperinflammatory, was left out; in the remaining {p1['n']}, the profile correlation was {p1_r2} and the "
          f"risk ratio for in-hospital death {sig3(p1['rr'][0])}."))

# Results: model fit
E.append(('Results', f"the mean log-likelihood per observed value was −{d2(-ft['ll_mean'])} (−{d2(-ft['ll_mean_wuhan'])} in the derivation "
          f"cohort), {ft['below_p5_n']} children ({pct(ft['below_p5_n'], co['n'])}%) were below the derivation cohort's 5th percentile of this "
          f"measure, the mean "
          f"largest posterior probability was {d2(ft['post_mean'])} ({d2(ft['post_mean_wuhan'])}), and {ft['post_below_08_n']} children "
          f"({pct(ft['post_below_08_n'], co['n'])}%) had a largest posterior probability below 0.8"))

# Discussion and conclusions
sens_pct, spec_pct = round(bp['sens'][0]), round(bp['spec'][0])
E.append(('Discussion', f"the frozen model assigned {d1(cl['S1']['pct'])}% of children with acute encephalitis or encephalopathy to this subphenotype."))
E.append(('Discussion', f"Their profile closely matched the derivation profile (r = {r_prim[0]}), their IL-10 and IL-6 were "
          f"{round(i10[0])}-fold and {d1(i6[0])}-fold higher than in the least abnormal subphenotype"))
assert round(rr[0]) == 4 and round(bp['rr'][0]) == 3
E.append(('Discussion', f"and a three-test bedside rule identified them with {sens_pct}% sensitivity and {spec_pct}% specificity."))
E.append(('Discussion', f"A model fitted in PIC alone, independent of the derivation model, also contained a class with this profile "
          f"(r = {d2(s1c[0]['r_s1'])}), and that class held all {cl['S1']['n']} children"))
E.append(('Discussion', f"({d1(cl['S1']['pct'])}% vs {d1(wr['S1_pct'])}%) but about as common as among derivation children admitted to the PICU "
          f"({d1(wr['S1_pct_picu'])}%)."))
w10 = val['IL-10|S1 vs S6 (least abnormal)']['wuhan_modal']
w6 = val['IL-6|S1 vs S6 (least abnormal)']['wuhan_modal']
assert i10[0] >= w10 and i6[0] < w6
E.append(('Discussion', f"({round(i10[0])}-fold vs {round(w10)}-fold) and IL-6 less ({d1(i6[0])}-fold vs {round(w6)}-fold)"))
prof = N['profiles']['Lactate']
assert all(prof[f'pic_{c}'] < prof[f'wuhan_{c}'] for c in ['S1', 'S2', 'S3', 'S4', 'S5', 'S6'])
E.append(('Discussion', 'Lactate was lower in all six PIC subphenotypes than in their derivation profiles'))
E.append(('Discussion', f"was missing for {round(100 - 100 * im['Temperature'] / co['n'])}% of the children."))
E.append(('Discussion', f"identified the hyperinflammatory subphenotype with {d1(bp['sens'][0])}% sensitivity and {d1(bp['spec'][0])}% specificity "
          f"from three tests available within hours of admission, and rule-positive children had a threefold risk of in-hospital death."))
E.append(('Conclusions', f"where a three-test bedside rule identified it with {sens_pct}% sensitivity and {spec_pct}% specificity."))
low = {c: cl[c]['post_below_08_n'] for c in ['S1', 'S2', 'S3', 'S4', 'S5', 'S6']}
assert sum(low.values()) == ft['post_below_08_n']
E.append(('Discussion', f"The model fitted PIC less closely than the derivation cohort, but {sum(low[c] for c in ['S2', 'S3', 'S4', 'S5'])} of "
          f"the {ft['post_below_08_n']} children with a largest posterior probability below 0.8 were in the four middle subphenotypes, and "
          f"only {low['S1']} was hyperinflammatory."))
assert left == 1
E.append(('Discussion', "Contrary to the addendum, the cohort was drawn from all diagnosis records rather than from discharge diagnoses alone, "
          f"which added {'one' if left == 1 else left} child."))
EXPECTED = E

pd.DataFrame(EXPECTED, columns=['section', 'text']).to_csv(OUT / 'manuscript_numbers.csv', index=False, encoding='utf-8-sig', lineterminator='\n')
print(len(EXPECTED), 'fragments written to manuscript_numbers.csv')
if DOCX.is_file():
    try:
        import docx
        doc = docx.Document(str(DOCX))
        body = ' '.join(p.text for p in doc.paragraphs)
        body = re.sub(r'\s+', ' ', body)
        body = body.translate(str.maketrans({'\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"'}))   # Word's curly quotes
        missing = [(w, t_) for w, t_ in EXPECTED if t_ not in body]
        print(f'Manuscript.docx: {len(EXPECTED) - len(missing)} of {len(EXPECTED)} fragments found')
        for w, t_ in missing:
            print('  NOT FOUND', w, '|', t_)
    except ImportError:
        print('python-docx is not installed: the manuscript check is skipped')
else:
    print('Manuscript.docx not in the folder: the manuscript check is skipped')

## Figure 7 and its legend
Same code as the manuscript figure: data from the exact tables and the frozen model, style of Figures 1-6. The legend is written
from the same numbers.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.colors import to_rgb
from matplotlib.ticker import LogLocator, FuncFormatter, NullFormatter, MultipleLocator
from matplotlib.transforms import Bbox

NUM = json.loads((OUT / 'pic_numbers.json').read_text(encoding='utf-8'))
MODEL = PKG
FIG_OUT = OUT


def table(name):
    """An exact PIC table, every cell read as text (no float conversion, so the reported precision is kept)."""
    return pd.read_csv(LOC / name, encoding='utf-8-sig', dtype=str, keep_default_na=False)


T_CLS, T_PROF, T_VAL, T_CON = table('06_classes.csv'), table('07_profiles.csv'), table('08_validators_by_class.csv'), table('09_validator_contrasts.csv')
T_DEATH, T_RULE, T_SENS = table('10_death.csv'), table('11_bedside_rule.csv'), table('13_sensitivity.csv')
WREF, RULE = MODEL['wuhan_reference'], MODEL['bedside_rule']
PAR = NUM['_source']['run_info']['parameters']                   # pre-specified constants of the PIC run (MIN_R, MIN_S1, N_BOOT, windows)

CLS = list(MODEL['classes'])
K = len(CLS)
S1, S6 = CLS.index('S1'), CLS.index('S6')
CLS_LABEL = MODEL['class_labels']                                 # {'S1': 'hyperinflammatory', 'S6': 'least abnormal'}
ORDER = list(MODEL['indicators'])                                 # = FINAL of the derivation analysis (Figure 2A order)
IND_LABEL = MODEL['indicator_labels']
N_PIC = int(NUM['cohort']['n'])
N_DER = int(WREF['children'])


def ci3(s):
    """'90.9 (70.8-98.9)' -> ('90.9', '70.8', '98.9'), as text."""
    m = re.match(r'^\s*(-?[\d.]+)\s*\((-?[\d.]+)-(-?[\d.]+)\)\s*$', s)
    if not m:
        raise ValueError(f'not an estimate with CI: {s!r}')
    return m.groups()


def row(t, key_col, key):
    r_ = t[t[key_col] == key]
    if len(r_) != 1:
        raise KeyError(f'{key!r} not found once in column {key_col!r}')
    return r_.iloc[0]


# --------------------------------------------------------------------------------------------------------------------------------
# Data for each panel (with consistency checks between the parsed JSON and the exact tables)
# A. Profiles: indicator order of Figure 2A; PIC observed means and derivation model means (SD units of the derivation cohort).
prof = T_PROF.set_index('indicator')
labels_in_order = [IND_LABEL[c] for c in ORDER]
assert set(prof.index) == set(labels_in_order) and len(prof) == len(ORDER), 'profile table and model indicators differ'
prof = prof.loc[labels_in_order]
P = len(ORDER)
PROF = {(coh, c): prof[f'{coh}_{c}'].astype(float).values for coh in ('pic', 'wuhan') for c in CLS}
for k, c in enumerate(CLS):                                       # derivation profile = frozen model means
    assert np.allclose(PROF[('wuhan', c)], np.round(np.asarray(MODEL['means'][k], float), 3), atol=6e-4), f'{c}: model means differ'

# B. Profile correlation of each PIC subphenotype with its derivation counterpart; children per subphenotype.
cls_t = T_CLS.set_index('class').loc[CLS]
N_CLS = cls_t['children_n'].astype(int).values
R_TXT = [cls_t.loc[c, 'profile_r_vs_wuhan_same_class'] for c in CLS]
R_VAL = np.array([float(s) for s in R_TXT])
assert int(N_CLS.sum()) == N_PIC, 'children per subphenotype do not add up to the cohort'
for c, n_, s in zip(CLS, N_CLS, R_TXT):
    assert NUM['classes'][c]['n'] == n_ and np.isclose(NUM['classes'][c]['r'], float(s))
prim = row(T_SENS, 'analysis', 'Primary')
R1_TXT = ci3(prim['S1_profile_r_ci'])                             # S1: r with its 95% CI, 3 decimals
assert Decimal(R1_TXT[0]) == Decimal(R_TXT[S1]), 'S1 profile r differs between tables 06 and 13'
MIN_R, MIN_S1, N_BOOT = float(PAR['MIN_R']), int(PAR['MIN_S1']), int(PAR['N_BOOT'])

# C, D. IL-6 and IL-10: PIC geometric means and children measured by most likely subphenotype; derivation geometric means.
val = T_VAL.set_index('validator')
CYTO = {}
for lab_, key in [('IL-6', 'il6'), ('IL-10', 'il10')]:
    gm = np.array([float(val.loc[lab_, f'{c}_geometric_mean']) for c in CLS])
    nm = np.array([int(val.loc[lab_, f'{c}_measured_n']) for c in CLS])
    der = np.array([float(WREF['validators'][key]['geometric_mean_by_class'][c]) for c in CLS])
    for k, c in enumerate(CLS):
        assert NUM['validator_gm'][lab_][c] == [nm[k], gm[k]]
        assert np.isclose(NUM['wuhan']['validators'][key]['gm'][c], der[k])
    con = row(T_CON[T_CON.validator == lab_], 'contrast', 'S1 vs S6 (least abnormal)')
    CYTO[lab_] = {'gm': gm, 'n': nm, 'der': der, 'ratio': ci3(con['ratio_of_geometric_means_ci']), 'ratio_der': con['wuhan_ratio_modal'],
                  'n1': int(con['S1_measured_n']), 'n0': int(con['comparison_measured_n'])}

# E. In-hospital death by subphenotype: PIC deaths, children and exact 95% CIs; derivation risk.
dth = T_DEATH.set_index('row').loc[CLS]
D_N, D_K = dth['children_n'].astype(int).values, dth['deaths_n'].astype(int).values
D_CI = np.array([[float(v) for v in ci3(s)] for s in dth['estimate_ci']])
D_DER = np.array([float(WREF['in_hospital_death']['risk_pct_by_class'][c]) for c in CLS])
assert np.array_equal(D_N, N_CLS) and int(D_K.sum()) == int(NUM['cohort']['deaths_n'])
assert np.allclose(D_DER, dth['wuhan_in_hospital_death_pct'].astype(float).values)
for k, c in enumerate(CLS):
    assert NUM['death'][c]['deaths'] == D_K[k] and np.allclose(NUM['death'][c]['pct_ci'], D_CI[k])
RR_TXT = ci3(row(T_DEATH, 'row', 'Risk ratio, S1 vs all other subphenotypes')['estimate_ci'])

# F. Bedside rule: PIC (external) and derivation (apparent) sensitivity, specificity, PPV and NPV with exact 95% CIs.
MEAS = [('Sensitivity (%)', 'sensitivity', 'Sensitivity'), ('Specificity (%)', 'specificity', 'Specificity'),
        ('Positive predictive value (%)', 'ppv', 'PPV'), ('Negative predictive value (%)', 'npv', 'NPV')]
WP = RULE['wuhan_performance']['all']
F_PIC = np.array([[float(v) for v in ci3(row(T_RULE, 'measure', m)['estimate_ci'])] for m, _, _ in MEAS])
F_DER = np.array([[100 * v for v in WP[key]] for _, key, _ in MEAS])
for (m, key, _), d_ in zip(MEAS, F_DER):                         # the derivation column of table 11 holds the same values
    assert np.allclose(d_, [float(v) for v in ci3(row(T_RULE, 'measure', m)['wuhan'])], atol=0.051)
RULE_N_PIC = int(row(T_RULE, 'measure', 'Children assessable (AST and LDH measured)')['value_n'])
RULE_N_DER = int(WP['assessable'])
assert RULE_N_PIC == NUM['bedside_pic']['assessable'] and RULE_N_DER == NUM['bedside_wuhan']['all']['assessable']
print('Consistency checks passed: pic_numbers.json, the exact tables and the frozen model agree.')

# --------------------------------------------------------------------------------------------------------------------------------
# Style: copied from analysis_v14/parts/a10_figures.py (font choice, rcParams, RYB and GREY at lines 30-42; the helpers _txt_color,
# panel_letters, g3, log_axis and class_axis unchanged), so Figure 7 matches Figures 1-6.
_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT = next((f for f in ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans'] if f in _fonts), 'sans-serif')
print('Figure font:', FONT if FONT == 'Arial' else f'{FONT} (Arial is not installed on this computer)')
plt.rcParams.update({'font.family': FONT, 'font.size': 11, 'axes.labelsize': 12, 'axes.labelweight': 'bold', 'xtick.labelsize': 11,
                     'ytick.labelsize': 11, 'legend.fontsize': 10.5, 'legend.handlelength': 1.6, 'legend.borderaxespad': 0.3, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': False, 'axes.linewidth': 0.9, 'xtick.major.width': 0.9, 'ytick.major.width': 0.9,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white', 'legend.frameon': False,
                     'pdf.fonttype': 42, 'ps.fonttype': 42, 'mathtext.fontset': 'custom', 'mathtext.rm': FONT,
                     'mathtext.it': f'{FONT}:italic', 'mathtext.bf': f'{FONT}:bold', 'mathtext.cal': FONT, 'mathtext.sf': FONT})
RYB = ['#C62828', '#EF6C00', '#E3A600', '#2A9D8F', '#4FA3D9', '#1F3F8F']        # red ... gold ... blue
assert K == 6, 'the RYB palette has six colors, one per subphenotype'
CC = RYB
RED, ORANGE, GOLD, TEAL, SKY, BLUE = RYB
GREY = '#8A8A8A'
INK = '#333333'                                                   # whiskers and neutral legend keys (as in bar_panel of Figure 3)


def _txt_color(rgb):
    r, g, b = to_rgb(rgb)
    return 'white' if 0.299 * r + 0.587 * g + 0.114 * b < 0.55 else 'black'


def panel_letters(fig, pairs):
    """Bold letters above the upper-left corner of each panel, aligned with the left edge of its y-axis title (or tick labels).
    An entry may carry a third element: an artist above the axes (such as a legend) that the letter is placed above."""
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    for ax, letter, *above in pairs:
        bb = ax.yaxis.get_tightbbox(rend) if ax.axison else ax.get_tightbbox(rend)
        x0 = inv.transform(((bb if bb is not None else ax.get_window_extent(rend)).x0, 0))[0]
        top = max(ax.get_window_extent(rend).y1, ax.xaxis.get_tightbbox(rend).y1 if ax.axison and ax.xaxis.get_tightbbox(rend) is not None
                  else 0, 0 if ax.axison else ax.get_tightbbox(rend).y1, *[a.get_window_extent(rend).y1 for a in above])
        y1 = inv.transform((0, top))[1]
        fig.text(x0, y1 + 0.08 / fig.get_figheight(), letter, fontsize=16, fontweight='bold', ha='left', va='bottom')


def g3(v):
    """Value label: 3 significant digits, trailing zeros kept (50.0, not 50); 2 below 1; thousands with a comma."""
    if v >= 1000:
        return f'{float(f"{v:.3g}"):,.0f}'
    return (f'{v:#.3g}' if v >= 1 else f'{v:#.2g}').rstrip('.')


def log_axis(ax, axis='y'):
    """Logarithmic axis with plain-number tick labels (1, 10, 100 or 1, 2, 5, 10 ...), unlabeled minor ticks."""
    a = ax.yaxis if axis == 'y' else ax.xaxis
    lo, hi = (ax.get_ylim() if axis == 'y' else ax.get_xlim())
    dec = np.log10(hi / lo) if lo > 0 else 3
    subs = (1.0,) if dec > 2.6 else ((1.0, 3.0) if dec > 1.3 else (1.0, 2.0, 5.0))
    a.set_major_locator(LogLocator(base=10, subs=subs))
    a.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}' if v >= 1000 else f'{v:g}'))
    a.set_minor_formatter(NullFormatter())


def class_axis(ax, counts=None, label='Subphenotype'):
    ax.set_xticks(range(K))
    ax.set_xticklabels([f'{c}\n({counts[i]})' if counts is not None else c for i, c in enumerate(CLS)])
    ax.set_xlim(-0.6, K - 0.4)
    if label:
        ax.set_xlabel(label)


def r2(s):
    """A correlation reported to 3 decimals, shown to 2 (round half up on the reported digits)."""
    return str(Decimal(s).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP))


def r3(s):
    return f'{Decimal(s):.3f}'


def sig3(s):
    """As in the manuscript text: 3 significant digits, at most 2 decimals, half up on the reported digits."""
    d = Decimal(str(s))
    return str(d.quantize(Decimal(1).scaleb(max(d.adjusted() - 2, -2)), rounding=ROUND_HALF_UP))


def label_points(ax, points, markers, extra=(), fs=10, gap=2.0):
    """Value labels that never touch a marker. points: (x, y, radius in points, text) to label; markers: (x, y, radius in points) of
    every marker in the panel. Each label goes above its point if that spot lies inside the axes and clear of every marker, of the
    artists in extra (such as the legend) and of the labels already written; otherwise below, right or left, whichever is free first."""
    fig_ = ax.figure
    fig_.canvas.draw()
    rend = fig_.canvas.get_renderer()
    pt = fig_.dpi / 72

    def mbox(x, y, r):
        cx, cy = ax.transData.transform((x, y))
        return Bbox.from_extents(cx - (r + 1) * pt, cy - (r + 1) * pt, cx + (r + 1) * pt, cy + (r + 1) * pt)

    taken = [mbox(*m) for m in markers] + [a.get_window_extent(rend) for a in extra]
    axbb = ax.get_window_extent(rend)
    for x, y, r, text in points:
        d = r + gap
        placed = None
        for off, ha, va in [((0, d), 'center', 'bottom'), ((0, -d), 'center', 'top'), ((d, 0), 'left', 'center'), ((-d, 0), 'right', 'center')]:
            t = ax.annotate(text, (x, y), xytext=off, textcoords='offset points', ha=ha, va=va, fontsize=fs)
            bb = t.get_window_extent(rend)
            inside = axbb.x0 <= bb.x0 and bb.x1 <= axbb.x1 and axbb.y0 <= bb.y0 and bb.y1 <= axbb.y1
            if inside and not any(bb.overlaps(o) for o in taken):
                placed = t
                break
            t.remove()
        if placed is None:                                        # no free spot: above the marker, and say so
            placed = ax.annotate(text, (x, y), xytext=(0, d), textcoords='offset points', ha='center', va='bottom', fontsize=fs)
            print(f'Note: no free spot for the label {text!r}; written above its marker')
        taken.append(placed.get_window_extent(rend))


def drawn_texts(fig_):
    """Every non-empty text that is drawn: figure texts, axes texts and labels, tick labels inside the view limits, legend texts."""
    out = list(fig_.texts)
    for ax in fig_.axes:
        out += list(ax.texts) + [ax.xaxis.label, ax.yaxis.label]
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            lo, hi = min(lim), max(lim)
            tol = 1e-9 * max(abs(lo), abs(hi), 1)
            out += [tk.label1 for tk in axis.get_major_ticks() if lo - tol <= tk.get_loc() <= hi + tol and tk.label1.get_visible()]
        if ax.get_legend() is not None:
            out += list(ax.get_legend().get_texts())
    return [t for t in out if t.get_visible() and t.get_text().strip()]


def text_overlaps(fig_):
    """Pairs of drawn texts whose boxes overlap by more than half a pixel (a layout check printed after saving)."""
    fig_.canvas.draw()
    rend = fig_.canvas.get_renderer()
    boxes = [(t.get_text(), t.get_window_extent(rend)) for t in drawn_texts(fig_)]
    return [(a, b) for i, (a, ba) in enumerate(boxes) for b, bb in boxes[i + 1:]
            if ba.x0 < bb.x1 - 0.5 and bb.x0 < ba.x1 - 0.5 and ba.y0 < bb.y1 - 0.5 and bb.y0 < ba.y1 - 0.5]


# Figure 2A labels and domain groups (a10_figures.py lines 195-196, 407 and 414; domains from a1_tools_data.py INDICATORS).
SHORT = {'Heart rate (z-score for age)': 'Heart rate, z-score', 'Respiratory rate (z-score for age)': 'Respiratory rate, z-score',
         'High-sensitivity CRP': 'CRP', 'Lymphocyte percentage': 'Lymphocytes, %', 'Neutrophil count': 'Neutrophils'}
SHORT2 = {**SHORT, 'Heart rate (z-score for age)': 'Heart rate', 'Respiratory rate (z-score for age)': 'Respiratory rate'}
DOMAIN_MEMBERS = {'Inflammation and blood count': ['neutrophils_abs', 'wbc', 'lymphocytes_pct', 'platelets', 'hemoglobin', 'hs_crp', 'procalcitonin'],
                  'Cell injury': ['alt', 'ast', 'ldh', 'ck', 'bilirubin_total', 'albumin'],
                  'Renal and metabolic': ['creatinine', 'urea', 'sodium', 'ammonia', 'lactate'],
                  'Coagulation': ['inr', 'aptt', 'fibrinogen', 'd_dimer'],
                  'Charted vital signs': ['temp_c', 'hr_z', 'rr_z']}
DOMAIN = {c: d for d, cs in DOMAIN_MEMBERS.items() for c in cs}
DOM_SHORT = {'Inflammation and blood count': 'Inflammation, blood count', 'Charted vital signs': 'Vital signs'}
assert all(c in DOMAIN for c in ORDER), 'an indicator of the model has no domain'

# --------------------------------------------------------------------------------------------------------------------------------
# Figure: 10 inches wide; panel positions in inches.
W7, H7 = 10.0, 10.2
fig = plt.figure(figsize=(W7, H7))


def ax_in(x, y, w, h):
    return fig.add_axes([x / W7, y / H7, w / W7, h / H7])


XL, XR = 0.92, 9.95                                               # left edge of the left panels, right edge of the right panels
axA = ax_in(XL, 7.3, XR - XL, 2.1)
PW, PH, PY2 = 2.5, 2.0, 3.7                                       # panels B-D (as Figure 3: 2.5 in wide at x = 0.92, 4.2, 7.45)
axB, axC, axD = (ax_in(x, PY2, PW, PH) for x in (XL, 4.2, 7.45))
axE = ax_in(XL, 0.72, 3.55, 2.0)
axF = ax_in(6.15, 0.72, XR - 6.15, 2.0)

# A. Standardized profiles of S1 and S6: derivation (model means; open markers, dashed) and PIC (observed means; filled, solid).
x = np.arange(P)
DASH = (0, (4.5, 2.5))
handlesA = []
for k in (S1, S6):
    col = CC[k]
    h_d, = axA.plot(x, PROF[('wuhan', CLS[k])], ls=DASH, color=col, lw=1.5, marker='o', ms=6, mfc='white', mec=col, mew=1.5, zorder=3)
    h_p, = axA.plot(x, PROF[('pic', CLS[k])], ls='-', color=col, lw=1.8, marker='o', ms=6.2, mfc=col, mec='white', mew=0.7, zorder=4)
    handlesA += [(h_d, f'{CLS[k]} derivation'), (h_p, f'{CLS[k]} PIC')]
h0 = axA.axhline(0, color=GREY, lw=1.0, zorder=1)
handlesA.append((h0, 'Derivation cohort mean'))
vals = np.concatenate([PROF[(coh, CLS[k])] for coh in ('pic', 'wuhan') for k in (S1, S6)])
axA.set_ylim(np.floor((vals.min() - 0.2) * 2) / 2, np.ceil((vals.max() + 0.2) * 2) / 2)     # half-SD steps clear of the markers
axA.yaxis.set_major_locator(MultipleLocator(1))
axA.set_xlim(-0.6, P - 0.4)
axA.set_xticks(x)
axA.set_xticklabels([SHORT2.get(IND_LABEL[c], IND_LABEL[c]) for c in ORDER], rotation=90, fontsize=10.5)
axA.set_ylabel('Standardized mean, SD')
doms = [DOMAIN[c] for c in ORDER]
starts = [j for j in range(P) if j == 0 or doms[j] != doms[j - 1]]
dom_texts = []
tr = axA.get_xaxis_transform()                                    # x in data units, y in axes fraction
for i_, j0 in enumerate(starts):
    j1 = starts[i_ + 1] if i_ + 1 < len(starts) else P
    axA.plot([j0 - 0.42, j1 - 0.58], [1.035, 1.035], transform=tr, color='#555555', lw=1.1, clip_on=False, solid_capstyle='butt')
    dom_texts.append(axA.text((j0 + j1 - 1) / 2, 1.06, DOM_SHORT.get(doms[j0], doms[j0]), transform=tr, ha='center', va='bottom',
                              fontsize=10.5, color='#333333'))

# B. Profile correlation with the derivation counterpart, by PIC subphenotype.
axB.bar(range(K), R_VAL, width=0.8, color=CC, alpha=0.88, zorder=2)          # wide enough to hold the value inside
hB = axB.axhline(MIN_R, color=INK, ls=(0, (4, 2.5)), lw=1.2, zorder=2.5)   # in front of the bars: a reference across all six
class_axis(axB, N_CLS, label=None)
axB.set_ylim(0, 1.2)
axB.set_yticks(np.linspace(0, 1, 6))
axB.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.1f}'))
axB.set_ylabel('Profile correlation r')
axB.legend([hB], [f'Replication criterion\n(r = {MIN_R:.2f})'], loc='upper right', fontsize=10, handlelength=2.2)

# C, D. IL-6 and IL-10 geometric means: derivation (open markers, left) and PIC (filled markers, right; value above).
DX, MS_D, MS_P = 0.2, 7.0, 7.5
R_D, R_P = MS_D / 2 + 0.8, MS_P / 2                               # marker radii in points, edge included
key_der = Line2D([], [], ls='none', marker='o', ms=MS_D, mfc='white', mec=INK, mew=1.5)
key_pic = Line2D([], [], ls='none', marker='o', ms=MS_P, mfc=INK, mec='white', mew=0.8)
for ax, lab_ in [(axC, 'IL-6'), (axD, 'IL-10')]:
    d_ = CYTO[lab_]
    for k in range(K):
        ax.plot(k - DX, d_['der'][k], 'o', ms=MS_D, mfc='white', mec=CC[k], mew=1.6, zorder=3)
        ax.plot(k + DX, d_['gm'][k], 'o', ms=MS_P, mfc=CC[k], mec='white', mew=0.8, zorder=4)
    allv = np.concatenate([d_['der'], d_['gm']])
    ax.set_yscale('log')
    ax.set_ylim(allv.min() / 1.8, allv.max() * 3.6)
    log_axis(ax)
    class_axis(ax, d_['n'], label=None)
    ax.set_ylabel(f'{lab_}, pg/mL')
    leg_ = ax.legend([key_der, key_pic], ['Derivation', 'PIC'], loc='upper right', fontsize=10, handletextpad=0.3)
    label_points(ax, [(k + DX, d_['gm'][k], R_P, g3(d_['gm'][k])) for k in range(K)],
                 [(k - DX, d_['der'][k], R_D) for k in range(K)] + [(k + DX, d_['gm'][k], R_P) for k in range(K)], extra=[leg_])

# E. In-hospital death: PIC (bars, exact 95% CI, deaths/children) and derivation (open markers, left of each bar).
XB, WB, XM = 0.12, 0.42, 0.24
for k in range(K):
    est, lo, hi = D_CI[k]
    axE.bar(k + XB, est, width=WB, color=CC[k], alpha=0.88, zorder=2)
    axE.errorbar(k + XB, est, yerr=[[est - lo], [hi - est]], fmt='none', ecolor=INK, elinewidth=1.1, capsize=3, zorder=3)
    axE.annotate(f'{D_K[k]}/{D_N[k]}', (k + XB, hi), xytext=(0, 2.5), textcoords='offset points', ha='center', va='bottom', fontsize=10)
    axE.plot(k - XM, D_DER[k], 'o', ms=7, mfc='white', mec=CC[k], mew=1.6, zorder=5, clip_on=False)
axE.set_ylim(0, D_CI[:, 2].max() * 1.2 + 6)
class_axis(axE, None)
axE.set_ylabel('In-hospital death, %')
axE.legend([key_der, Rectangle((0, 0), 1, 1, fc=GREY, alpha=0.88)], ['Derivation', 'PIC'], loc='upper right', fontsize=10,
           handletextpad=0.4)

# F. Bedside rule for S1: derivation (apparent; open grey) and PIC (external; filled black), estimates with exact 95% CIs.
DY = 0.16
for i, (der, pic) in enumerate(zip(F_DER, F_PIC)):
    axF.errorbar(der[0], i - DY, xerr=[[der[0] - der[1]], [der[2] - der[0]]], fmt='o', ms=7, mfc='white', mec=GREY, mew=1.6,
                 ecolor=GREY, elinewidth=1.6, capsize=3, zorder=3, clip_on=False)
    axF.errorbar(pic[0], i + DY, xerr=[[pic[0] - pic[1]], [pic[2] - pic[0]]], fmt='o', ms=7, mfc='black', mec='black', mew=1,
                 ecolor='black', elinewidth=1.6, capsize=3, zorder=4, clip_on=False)
    axF.annotate(f'{der[0]:.1f}', (der[0], i - DY), xytext=(0, 5.5), textcoords='offset points', ha='center', va='bottom', fontsize=10,
                 color='#555555')
    axF.annotate(f'{pic[0]:.1f}', (pic[0], i + DY), xytext=(0, -5.5), textcoords='offset points', ha='center', va='top', fontsize=10)
axF.set_yticks(range(len(MEAS)))
axF.set_yticklabels([m[2] for m in MEAS])
axF.set_ylim(len(MEAS) - 0.3, -0.7)
axF.tick_params(axis='y', length=0)
axF.set_xlim(50, 101.2)
axF.set_xticks(np.arange(50, 101, 10))
axF.set_xlabel('Estimate, %')
axF.set_ylabel('S1 bedside rule')
axF.legend([Line2D([], [], ls='-', marker='o', ms=7, mfc='white', mec=GREY, mew=1.6, color=GREY, lw=1.6),
            Line2D([], [], ls='-', marker='o', ms=7, mfc='black', mec='black', color='black', lw=1.6)],
           ['Derivation, apparent', 'PIC, external'], loc='lower left', fontsize=10, handlelength=2.0)

# Panel letters (A sits above the domain labels); the legend of panel A runs along the line of its letter.
panel_letters(fig, [(axA, 'A', *dom_texts), (axB, 'B'), (axC, 'C'), (axD, 'D'), (axE, 'E'), (axF, 'F')])
rend = fig.canvas.get_renderer()
letterA = next(t for t in fig.texts if t.get_text() == 'A')
bbA = letterA.get_window_extent(rend).transformed(fig.transFigure.inverted())
axA.legend([h for h, _ in handlesA], [l for _, l in handlesA], loc='center right', ncol=len(handlesA), handlelength=2.6,
           columnspacing=1.6, handletextpad=0.6, bbox_to_anchor=(XR / W7, (bbA.y0 + bbA.y1) / 2), bbox_transform=fig.transFigure,
           borderaxespad=0, borderpad=0)

# --------------------------------------------------------------------------------------------------------------------------------
# Save
OUT = FIG_OUT
fig.savefig(OUT / 'Figure7_external_validation.pdf', bbox_inches='tight')
fig.savefig(OUT / 'Figure7_external_validation.png', dpi=200, bbox_inches='tight')
fig.canvas.draw()
tb = fig.get_tightbbox(fig.canvas.get_renderer())
print(f'Saved figure, {tb.width + 0.2:.2f} x {tb.height + 0.2:.2f} inches (tight bounding box with 0.1-inch padding)')
clash = text_overlaps(fig)
print('Layout check: no overlapping text' if not clash else f'Layout check: overlapping text {clash}')
plt.close(fig)

# --------------------------------------------------------------------------------------------------------------------------------
# Legend (every number from the tables above)
crit = ', '.join(f"{c['test'].upper()} ≥{c['value']:g}" + (f" {c['unit']}" if c['unit'] != 'ratio' else '') for c in RULE['criteria'])
crit = ' and '.join(crit.rsplit(', ', 1))
vw = PAR['VALIDATOR_WINDOW_H']
others = ', '.join(r3(R_TXT[k]) for k in range(K) if k not in (S1,))
others = ' and '.join(others.rsplit(', ', 1))
i6, i10 = CYTO['IL-6'], CYTO['IL-10']
rr_der = WREF['in_hospital_death']['rr_S1_vs_others']
WORD = {1: 'one', 2: 'two', 3: 'three'}
LEGEND = (
    f"**Figure 7. External validation of the subphenotypes in the Paediatric Intensive Care (PIC) database.** "
    f"The frozen derivation model ({N_DER} children) assigned each of the {N_PIC} PIC children to the most likely subphenotype after "
    f"standardization with the derivation constants. In C–E, derivation values are by most likely subphenotype. "
    f"(A) Mean standardized values of the {P} indicators in S1 ({CLS_LABEL['S1']}) and S6 ({CLS_LABEL['S6']}): derivation model means "
    f"and PIC observed means, in derivation SD units. "
    f"(B) Pearson correlation across the {P} indicators between the mean profile of each PIC subphenotype and its derivation profile: "
    f"S1, {R1_TXT[0]} (95% CI {R1_TXT[1]} to {R1_TXT[2]}; Fisher z with the SD of {N_BOOT:,} bootstrap resamples); S2 to S{K}, "
    f"{others}. The replication criterion applied to S1: at least {MIN_S1} children and r ≥ {MIN_R:.2f}. Numbers in parentheses, "
    f"PIC children. "
    f"(C, D) Geometric means of the first IL-6 and IL-10 from {abs(vw[0])} h before to {vw[1]} h after admission (logarithmic axes; "
    f"numbers next to filled markers, PIC values; in parentheses, PIC children measured). S1 versus S6, ratio of geometric means: "
    f"IL-6 {sig3(i6['ratio'][0])} (95% percentile-bootstrap CI {sig3(i6['ratio'][1])} to {sig3(i6['ratio'][2])}; derivation "
    f"{sig3(i6['ratio_der'])}); IL-10 {sig3(i10['ratio'][0])} ({sig3(i10['ratio'][1])} to {sig3(i10['ratio'][2])}; derivation "
    f"{sig3(i10['ratio_der'])}). "
    f"(E) In-hospital death with exact 95% CIs; deaths/children above the bars. S1 versus the other subphenotypes, risk ratio "
    f"{sig3(RR_TXT[0])} ({sig3(RR_TXT[1])} to {sig3(RR_TXT[2])}; derivation {sig3(rr_der)}). "
    f"(F) Accuracy of the S1 bedside rule (at least {WORD[RULE['positive_if_at_least']]} of {crit}) against the model's assignment, with "
    f"exact 95% CIs, in children with AST and LDH measured: derivation, apparent ({RULE_N_DER} of {N_DER} children); PIC, external "
    f"({RULE_N_PIC} of {N_PIC}). PPV and NPV, positive and negative predictive values.")
n_words = len(re.sub(r'\*\*', '', LEGEND).split())
(OUT / 'figure7_legend.md').write_text(LEGEND + '\n', encoding='utf-8', newline='\n')
print(f'Legend written: {n_words} words')
if n_words >= 300:
    print('WARNING: the legend has 300 words or more')

FIG_DATA = {'profiles': {f'{a}_{b}': list(map(float, v)) for (a, b), v in PROF.items()}, 'r': list(map(float, R_VAL)),
            'cytokines': {k: {kk: [float(x) for x in vv] for kk, vv in v.items() if kk in ('gm', 'n', 'der')} for k, v in CYTO.items()},
            'death_ci': D_CI.tolist(), 'death_der': D_DER.tolist(), 'rule_pic': F_PIC.tolist(), 'rule_der': F_DER.tolist()}

## Additional file 2, Tables S27-S33
Appended to a copy of the submitted workbook (Tables S1-S26) when it is in the folder, in the same plain layout; otherwise
written to a new workbook with a contents sheet.

In [ ]:
import shutil
P1 = SUB03_LOCAL
AF2_OUT = OUT / 'Additional_file_2_supplementary_tables.xlsx'
FROM_SUBMITTED = AF2_IN.is_file()
if FROM_SUBMITTED:
    shutil.copyfile(AF2_IN, AF2_OUT)
    wb = openpyxl.load_workbook(AF2_OUT)
    assert wb.sheetnames[-1] == 'Table S26', 'the submitted workbook should end with Table S26'
else:
    wb = openpyxl.Workbook()
    c0 = wb.active
    c0.title = 'Contents'
    c0.cell(1, 1, 'Additional file 2. Supplementary tables')
    c0.cell(4, 1, 'Sheet')
    c0.cell(4, 2, 'Title')
    print('Additional_file_2_supplementary_tables.xlsx (Tables S1-S26) not found: writing Tables S27-S33 only')


CLASSNAME = {'S1': 'S1 hyperinflammatory', 'S2': 'S2 systemic inflammatory', 'S3': 'S3 lymphocyte-predominant',
             'S4': 'S4 neutrophilic acute-phase', 'S5': 'S5 low metabolic stress', 'S6': 'S6 least abnormal'}
YESNO = {'True': 'yes', 'False': 'no'}


def t(name, folder=LOC):
    return pd.read_csv(folder / name, dtype=str, keep_default_na=False, encoding='utf-8-sig')


def dec(s, d):
    """A number written by the notebook (trailing zeros dropped) shown with d decimals; text and blanks unchanged."""
    try:
        x = float(s)
    except ValueError:
        return s
    out = f'{x:.{d}f}'
    return out[1:] if out.startswith('-') and float(out) == 0 else out


def write(sheet, title, blocks, note=None):
    """blocks: list of (subheading or None, DataFrame)."""
    ws = wb.create_sheet(sheet)
    ws.cell(1, 1, f'{sheet}. {title}')
    r = 3
    for sub, df in blocks:
        if sub:
            ws.cell(r, 1, sub)
            r += 1
        for j, c in enumerate(df.columns, 1):
            ws.cell(r, j, c)
        for i, row in enumerate(df.itertuples(index=False), 1):
            for j, v in enumerate(row, 1):
                if v != '':
                    ws.cell(r + i, j, v)
        r += len(df) + 2
    if note:
        ws.cell(r, 1, note)


rows = []
SG_OLD = 'S-g: hs-CRP below a limit entered at the PIC limit (no Wuhan floor)'
SG_NEW = "S-g: hs-CRP below a limit entered at the PIC limit (without the derivation laboratory's reporting floor)"

# S27 cohort
flow = t('02_flow.csv').rename(columns={'count_n': 'count'})
coh = t('03b_cohort.csv')
coh['measure'] = coh.measure.map({'children_n': 'Eligible children', 'age_years_median_iqr': 'Age, years, median (IQR)',
                                  'male_n': 'Boys', 'male_pct': 'Boys, %', 'in_hospital_deaths_n': 'In-hospital deaths',
                                  'in_hospital_deaths_pct': 'In-hospital deaths, %', 'first_icu_unit_picu_n': 'First intensive care unit: PICU',
                                  'first_icu_unit_picu_pct': 'First intensive care unit: PICU, %',
                                  'blood_tests_of_22_median_iqr': 'Blood tests measured of 22, median (IQR)',
                                  'missing_pct_24_indicators': 'Missing values of the 24 indicators, %'})
assert coh.measure.notna().all()
cg = t('03_code_groups.csv').rename(columns={'code_group': 'diagnosis code group (index admission; first match in this order)',
                                             'children_n': 'children', 'children_pct': 'children, %'})
write('Table S27', 'External validation in PIC: cohort flow, characteristics and diagnosis code groups',
      [('Cohort flow', flow), ('Eligible children', coh), ('Diagnosis code groups', cg)],
      'PIC, Paediatric Intensive Care database (version 1.1.0). Inclusion and exclusion codes in Additional file 1.')
rows.append(('Table S27', 'External validation in PIC: cohort flow, characteristics and diagnosis code groups',
             'PIC_SUB_02: 02_flow, 03b_cohort, 03_code_groups'))

# S28 availability, scale and items
av = t('05_availability.csv')
av['in_model'] = av.in_model.map(YESNO)
av = av.rename(columns={'measure': 'test', 'in_model': 'model indicator', 'measured_n': 'PIC measured', 'measured_pct': 'PIC measured, %',
                        'wuhan_measured_pct': 'derivation measured, %', 'pic_median_iqr_wuhan_unit': 'PIC median (IQR), derivation unit',
                        'wuhan_median_iqr': 'derivation median (IQR)'})
sc = t('12b_indicator_scale.csv').rename(columns={'measured_n': 'measured', 'measured_pct': 'measured, %',
                                                 'wuhan_measured_pct': 'derivation measured, %', 'standardized_mean': 'standardized mean',
                                                 'standardized_sd': 'standardized SD', 'at_lower_winsor_limit_n': 'at the lower winsorizing limit',
                                                 'at_upper_winsor_limit_n': 'at the upper winsorizing limit'})
it = t('04_items.csv').rename(columns={'concept': 'test (code name)', 'itemid': 'PIC ITEMID', 'label': 'PIC label',
                                      'rows_eligible_children_n': 'results (eligible children)', 'units_normalized': 'unit',
                                      'rows_unit_not_convertible_n': 'unit not convertible', 'rows_censored_n': 'censored results',
                                      'censored_pct': 'censored, %', 'linked_without_admission_number_n': 'linked without admission number'})
write('Table S28', 'External validation in PIC: availability, distributions and scale of the indicators, and the PIC laboratory items',
      [('Availability and distributions', av), ('Indicators on the derivation scale', sc), ('PIC laboratory items', it)],
      'Vital signs come from intensive care charting in PIC.')
rows.append(('Table S28', 'External validation in PIC: availability, distributions and scale of the indicators, and the PIC laboratory items',
             'PIC_SUB_02: 05_availability, 12b_indicator_scale, 04_items'))

# S29 subphenotypes and model fit
cl = t('06_classes.csv')
cl['class'] = cl['class'].map(CLASSNAME)
cl['profile_r_vs_wuhan_same_class'] = cl['profile_r_vs_wuhan_same_class'].map(lambda s: dec(s, 3))
cl = cl.drop(columns=['label']).rename(columns={
    'class': 'subphenotype (frozen model)', 'children_n': 'children', 'children_pct': 'children, %',
    'estimated_share_pct': 'estimated share (mean posterior), %', 'mean_posterior_assigned': 'mean posterior probability of the assigned class',
    'posterior_below_0_8_n': 'children with largest posterior < 0.8', 'posterior_ge_0_8_pct': 'children with largest posterior >= 0.8, %',
    'profile_r_vs_wuhan_same_class': 'profile correlation with the derivation class (r)', 'wuhan_children_pct': 'derivation cohort, %',
    'wuhan_picu_children_pct': 'derivation children with a PICU stay, %'})
fit = t('12_model_fit.csv').rename(columns={'pic': 'PIC', 'wuhan': 'derivation cohort'})
FIT = {'Log-likelihood per observed value, mean': 'Log-likelihood per observed value, mean',
       'Log-likelihood per observed value, median': 'Log-likelihood per observed value, median',
       'Children below the Wuhan 5th percentile of log-likelihood per value (children, rounded)':
           "Children below the derivation cohort's 5th percentile of the log-likelihood per observed value",
       'Largest posterior probability, mean': 'Largest posterior probability, mean',
       'Largest posterior probability below 0.8 (children, rounded)': 'Children with a largest posterior probability below 0.8',
       'Missing indicator values (%)': 'Missing indicator values, %'}
fit['measure'] = fit.measure.map(FIT)
assert fit.measure.notna().all()
write('Table S29', "External validation in PIC: subphenotypes assigned by the frozen model and the model's fit",
      [('Subphenotypes', cl), ('Model fit', fit)],
      'Profile correlation: Pearson r across the 24 indicators between the mean standardized profile in PIC and the derivation '
      'model means of the same class.')
rows.append(('Table S29', "External validation in PIC: subphenotypes assigned by the frozen model and the model's fit",
             'PIC_SUB_02: 06_classes, 12_model_fit'))

# S30 profiles
pr = t('07_profiles.csv')
for c in pr.columns:
    if c.startswith('pic_S'):
        pr[c] = pr[c].map(lambda s: dec(s, 2))
    elif c.startswith('wuhan_S'):
        pr[c] = pr[c].map(lambda s: dec(s, 3))
pr = pr.rename(columns={'pic_measured_pct': 'PIC measured, %'})
pr.columns = [c.replace('pic_', 'PIC ').replace('wuhan_', 'derivation ') for c in pr.columns]
write('Table S30', 'External validation in PIC: mean standardized profiles by subphenotype (PIC, observed; derivation, model means)',
      [(None, pr)], 'Standardized with the derivation constants (SD units of the derivation cohort). PIC means are shown when at '
      'least 5 children in the class were measured.')
rows.append(('Table S30', 'External validation in PIC: standardized profiles by subphenotype, PIC and derivation', 'PIC_SUB_02: 07_profiles'))

# S31 validators
vb = t('08_validators_by_class.csv')
vb.columns = [c.replace('_measured_n', ' measured').replace('_geometric_mean', ' geometric mean').replace('measured_n', 'measured')
              .replace('measured_pct', 'measured, %') for c in vb.columns]
vc = t('09_validator_contrasts.csv').rename(columns={'S1_measured_n': 'S1 measured', 'comparison_measured_n': 'comparison measured',
                                                    'ratio_of_geometric_means_ci': 'ratio of geometric means (95% bootstrap CI)',
                                                    'wuhan_ratio_modal': 'derivation cohort, by most likely class'})
assert list(vc.columns[:2]) == ['validator', 'contrast'], vc.columns
write('Table S31', 'External validation in PIC: cytokines and ferritin by subphenotype (most likely class)',
      [('Geometric means (pg/mL; ferritin ng/mL)', vb), ('Contrasts', vc)],
      'First value from 24 h before to 48 h after admission; results of exactly 0 entered as half the smallest positive value. '
      'Geometric means are shown when at least 5 children in the class were measured.')
rows.append(('Table S31', 'External validation in PIC: cytokines and ferritin by subphenotype',
             'PIC_SUB_02: 08_validators_by_class, 09_validator_contrasts'))

# S32 death and bedside rule
de = t('10_death.csv')
de['row'] = de.row.map(lambda s: CLASSNAME.get(s, s))
de = de.rename(columns={'row': 'subphenotype or comparison', 'children_n': 'children', 'deaths_n': 'in-hospital deaths',
                        'risk_pct': 'risk, %', 'estimate_ci': 'estimate (95% CI)', 'wuhan_in_hospital_death_pct': 'derivation: in-hospital death, %',
                        'wuhan_death_or_critical_discharge_pct': 'derivation: death or critical discharge, %'})
br = t('11_bedside_rule.csv').rename(columns={'value_n': 'children', 'estimate_ci': 'estimate (95% CI)', 'wuhan': 'derivation cohort (apparent)'})
write('Table S32', 'External validation in PIC: in-hospital death by subphenotype and the S1 bedside rule',
      [('In-hospital death', de), ('Bedside rule: at least 2 of AST >= 200 U/L, LDH >= 700 U/L, INR >= 1.5', br)],
      'Risk ratios with Katz CIs; proportions with exact (Clopper-Pearson) CIs; odds ratio adjusted for age (restricted cubic spline, '
      "3 knots). The rule is assessed against the frozen model's assignment; a child is assessable when AST and LDH were measured.")
rows.append(('Table S32', 'External validation in PIC: in-hospital death and the S1 bedside rule', 'PIC_SUB_02: 10_death, 11_bedside_rule'))

# S33 sensitivity analyses, P-1 and de novo LPA
se = t('13_sensitivity.csv')
p1 = t('15_posthoc_discharge_only.csv', P1).iloc[[1]]
assert p1.analysis.iloc[0].startswith('P-1') and se.analysis.iloc[0] == 'Primary'
se = pd.concat([se, p1], ignore_index=True).fillna('')
assert (se.analysis == SG_OLD).sum() == 1
se['analysis'] = se.analysis.replace(SG_OLD, SG_NEW)
se = se.rename(columns={'children_n': 'children', 'S1_n': 'S1 children', 'S1_pct': 'S1, %',
                        'S1_profile_r_ci': 'S1 profile r (95% CI)', 'replication_criterion_met': 'replication criterion met',
                        'il6_ratio_S1_vs_S6': 'IL-6 ratio S1 vs S6 (95% CI)', 'il10_ratio_S1_vs_S6': 'IL-10 ratio S1 vs S6 (95% CI)',
                        'death_rr_S1_vs_others': 'in-hospital death, risk ratio S1 vs others (95% CI)', 'kappa_vs_primary': 'kappa vs primary',
                        'common_children_n': 'children in both'})
pc = t('15b_posthoc_counts.csv', P1)
PC = {'children_primary_analysis_n': 'Children in the primary analysis',
      'children_with_inclusion_code_among_discharge_diagnoses_n': 'Inclusion code among the discharge diagnoses',
      'children_with_inclusion_code_only_among_admitting_or_clinic_diagnoses_n': 'Inclusion code only among admitting or clinic diagnoses (left out)',
      'S1_children_left_out_n': 'S1 children left out', 'in_hospital_deaths_left_out_n': 'In-hospital deaths left out',
      'left_out_group_encephalopathy_n': 'Left out: encephalopathy codes',
      'left_out_group_bacterial_meningoencephalitis_n': 'Left out: bacterial meningoencephalitis',
      'left_out_group_adem_n': 'Left out: ADEM or disseminated encephalitis',
      'left_out_group_viral_encephalitis_n': 'Left out: viral encephalitis',
      'left_out_group_brainstem_or_focal_n': 'Left out: brainstem or focal encephalitis, encephalomyelitis',
      'left_out_group_unspecified_encephalitis_n': 'Left out: meningoencephalitis or encephalitis, unspecified'}
pc['measure'] = pc.measure.map(PC)
assert pc.measure.notna().all()
pc = pc.rename(columns={'measure': 'children', 'value': 'n'})
dn = t('14_de_novo_lpa.csv')
sel = dn[dn.classes.str.fullmatch(r'\d+')][['classes', 'bic', 'smallest_class_n', 'meets_5pct_rule', 'starts',
                                            'starts_reaching_best', 'chosen']].copy()
for c in ['meets_5pct_rule', 'chosen']:
    sel[c] = sel[c].map(YESNO)
sel = sel.rename(columns={'bic': 'BIC', 'smallest_class_n': 'smallest class (children)', 'meets_5pct_rule': 'every class >= 5%',
                          'starts': 'random starts', 'starts_reaching_best': 'starts reaching the best log-likelihood'})
ch = dn[dn.classes.str.startswith('chosen solution')][['classes', 'children_n', 'r_with_wuhan_S1', 'closest_wuhan_class',
                                                      'r_with_closest_wuhan_class', 'also_assigned_S1_by_frozen_model_n']].copy()
ch['classes'] = ch.classes.str.replace('chosen solution, ', '', regex=False)
ch['closest_wuhan_class'] = ch.closest_wuhan_class.map(CLASSNAME)
ch = ch.rename(columns={'classes': 'class (chosen solution)', 'children_n': 'children', 'r_with_wuhan_S1': 'r with derivation S1',
                        'closest_wuhan_class': 'closest derivation class', 'r_with_closest_wuhan_class': 'r with the closest derivation class',
                        'also_assigned_S1_by_frozen_model_n': 'children assigned S1 by the frozen model'})
s1like = dn[dn.classes.str.startswith('S1-like')]
assert len(s1like) == 1 and s1like.closest_wuhan_class.iloc[0] == 'yes'
write('Table S33', 'External validation in PIC: sensitivity analyses, post hoc analysis P-1 and de novo latent profile analysis (S-f)',
      [('Sensitivity analyses (S-a to S-g pre-specified; P-1 post hoc)', se), ('P-1: children left out', pc),
       ('S-f: de novo latent profile analysis in PIC, 1 to 6 classes (lowest BIC with every class >= 5%)', sel),
       ('S-f: classes of the chosen solution', ch)],
      f'Not estimable: fewer than 5 children measured in a group, or no deaths in a group. S-f: the chosen solution has an S1-like '
      f'class (r >= 0.80 with the derivation S1 profile; r = {s1like.r_with_wuhan_S1.iloc[0]}).')
rows.append(('Table S33', 'External validation in PIC: sensitivity analyses, post hoc analysis P-1 and de novo latent profile analysis',
             'PIC_SUB_02: 13_sensitivity, 14_de_novo_lpa; PIC_SUB_03: 15_posthoc_discharge_only, 15b_posthoc_counts'))

# No leftover notebook labels for the derivation cohort
for name in [r[0] for r in rows]:
    for row in wb[name].iter_rows(values_only=True):
        for v in row:
            assert v is None or 'Wuhan' not in str(v), (name, v)

# Contents
c = wb['Contents']
c.cell(2, 1, 'Every table of the analysis, one per sheet; Tables S27 to S33 are the external validation in PIC. BCH, bias-corrected '
       'three-step estimation; CI, confidence interval; ITES, infection-triggered encephalopathy syndrome; PIC, Paediatric Intensive '
       'Care database; SD, standard deviation.')
c.cell(4, 3, 'Same table in the notebook outputs (output_subphenotypes/tables; PIC: exact tables of PIC_SUB_02 and PIC_SUB_03)')
r0 = c.max_row
assert c.cell(r0, 1).value == ('Table S26' if FROM_SUBMITTED else 'Sheet')
for i, (sh, title, src) in enumerate(rows, 1):
    c.cell(r0 + i, 1, sh)
    c.cell(r0 + i, 2, title)
    c.cell(r0 + i, 3, src)
wb.save(AF2_OUT)
print('saved', AF2_OUT.name, 'with', wb.sheetnames[-7], 'to', wb.sheetnames[-1])

## Summary and comparison with the manuscript build
Each result is compared with the manuscript build by a SHA-256 of its content (numbers and text, not file bytes, so fonts,
dates and line endings do not matter).

In [ ]:
def h(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()


wb2 = openpyxl.load_workbook(OUT / 'Additional_file_2_supplementary_tables.xlsx')
pic_sheets = {n: [[v for v in r] for r in wb2[n].iter_rows(values_only=True)] for n in [f'Table S{i}' for i in range(27, 34)]}
N_saved = json.loads((OUT / 'pic_numbers.json').read_text(encoding='utf-8'))
N_saved.pop('_source')
GOT = {'pic_numbers': h(N_saved), 'manuscript_numbers': h(EXPECTED), 'figure7_data': h(FIG_DATA),
       'figure7_legend': h(LEGEND), 'tables_S27_S33': h(pic_sheets)}
EXPECTED_HASHES = {
 "pic_numbers": "4b8aa21eb251bb87dd8487cd34ea54eed3b82771bba24ec83d68946326e44e0a",
 "manuscript_numbers": "63aa946af9e345596e314245d438f3e7a7b3a674e2617aaa1171819f854c963f",
 "figure7_data": "2268c6c92bb929b3d7ddbb888bfbea08a8cf4d962c9d16f39d17b72fb27e5f9b",
 "figure7_legend": "02a1017afb9f75ced098542bbf2d18e6dc8560dd5444e811ad3f3bf28537dbc8",
 "tables_S27_S33": "a2ef8a116af1c00cb4d5db37b9c106f210595676d07a78fadc3e470f938ff1bd"
}
for k, v in GOT.items():
    print(f"{k:20s} {'MATCH' if EXPECTED_HASHES.get(k) == v else 'DIFFERENT'}  {v[:16]}")
info = {'notebook': 'PIC_SUB_04_manuscript_outputs', 'inputs': {'PIC_SUB_02': FP_SUB02, 'PIC_SUB_03': FP_SUB03, 'frozen_model': MODEL_SHA},
        'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__,
        'openpyxl': openpyxl.__version__, 'figure_font': FONT, 'content_sha256': GOT,
        'matches_manuscript_build': all(EXPECTED_HASHES.get(k) == v for k, v in GOT.items())}
try:
    import scipy
    info['scipy'] = scipy.__version__
except ImportError:
    info['scipy'] = None
info['same_environment_as_pic_sub02'] = (platform.python_version(), pd.__version__, np.__version__) == (RUN['python'], RUN['pandas'], RUN['numpy'])
(OUT / 'run_info_sub04.json').write_text(json.dumps(info, indent=1), encoding='utf-8', newline='\n')
print('SciPy in this environment:', info['scipy'])
print('Same Python, pandas and NumPy as the PIC_SUB_02 run:', 'yes, so this SciPy version is the one to report in Methods (Software)'
      if info['same_environment_as_pic_sub02'] else 'no; report the SciPy version of the environment that ran PIC_SUB_02')
print('Outputs:', ', '.join(sorted(p.name for p in OUT.iterdir())))